[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Agregación, relaciones entre variables y visualización efectiva.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete
**Jueves 22 de octubre de 2026 · 6:30 a 8:30 pm (CDMX)** · Módulo II: *Describir y comparar*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En esta sesión se resumirán datos por grupos (`groupby`, `pivot_table`, `crosstab`, `melt`), se medirán relaciones entre variables con las correlaciones de Pearson y Spearman sin confundir correlación con causalidad (confusores y paradoja de Simpson) y se construirán gráficas claras con *Matplotlib* y *Seaborn*. Al final se presenta la **Evaluación semanal 1**.

## 🎯 Objetivo de la sesión.

**Resumir por grupos, medir relaciones entre variables y comunicarlas con gráficas claras, sin confundir correlación con causalidad.**

Al terminar podrás:
1. Agregar con `groupby`/`agg`, `pivot_table` y `crosstab`, y cambiar entre formato ancho y largo con `melt`.
2. Calcular e interpretar las correlaciones de Pearson y Spearman y leer tablas de contingencia.
3. Reconocer confusores y la paradoja de Simpson.
4. Elegir la gráfica adecuada y cuidar escalas, facetas, color y anotaciones.

## 🗺️ Agenda (120 min).

| Min | Bloque |
|---|---|
| 10 | Repaso |
| 25 | 1. Agregación |
| 25 | 2. Relaciones entre variables |
| 30 | 3. Visualización efectiva |
| 20 | 4. Ejercicio de la sesión |
| 10 | 5. Presentación de la Evaluación semanal 1 |

🟢 = ejercicio básico · 🔴 = reto opcional

### Preparación del entorno.

* La siguiente celda importará las bibliotecas, incluido el módulo `stats` de *SciPy*, y definirá `DATA` y la semilla `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats

warnings.filterwarnings("ignore", message="vert: bool was deprecated")

**Nota:** el filtro de advertencias oculta solo un aviso interno de *Seaborn* 0.13 con *Matplotlib* 3.11 al dibujar diagramas de caja (`vert: bool was deprecated`). No afecta los resultados.

---
## 0. Repaso (10 min).

En la sesión 3 se vio que la masa y la aleta de los pingüinos se describen mal en conjunto y bien **por especie**. Hoy se formaliza esa idea: resumir por grupos y estudiar cómo se relacionan las variables. Se usarán `penguins_limpio` (se regenera si no existe) y `tips`.

* La siguiente celda leerá `penguins_limpio.csv` o lo regenerará desde `penguins.csv` si no existe.

In [ ]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio).astype({"species": "category", "island": "category", "sex": "category"})
pinguinos.shape

`tips` registra 244 cuentas de un restaurante: `total_bill` (cuenta, en dólares), `tip` (propina), `sex` y `smoker` de quien paga, `day`, `time` (comida o cena) y `size` (personas en la mesa). Se creará la **propina porcentual** `tip_pct = 100 · tip / total_bill` y se ordenarán los días.

* La siguiente celda leerá `tips.csv`, creará `tip_pct` y convertirá `day` y `time` en categorías ordenadas.

In [ ]:
propinas = (
    pd.read_csv(os.path.join(DATA, "tips.csv"))
    .assign(tip_pct=lambda d: 100 * d["tip"] / d["total_bill"])
    .astype({"day": pd.CategoricalDtype(["Thur", "Fri", "Sat", "Sun"], ordered=True),
             "time": pd.CategoricalDtype(["Lunch", "Dinner"], ordered=True)})
)
propinas.head()

---
## 1. Agregación (25 min).

| Herramienta | Para qué | Resultado |
|---|---|---|
| `groupby(...).agg(...)` | Resumir una o varias columnas por grupo | Una fila por grupo |
| `pivot_table` | Resumen con un grupo en filas y otro en columnas | Tabla de doble entrada |
| `crosstab` | Conteos (o proporciones) de dos categóricas | Tabla de contingencia |
| `melt` | Pasar de formato **ancho** (una columna por variable) a **largo** (una fila por observación y variable) | Ideal para *Seaborn* y facetas |

**Nota:** en *pandas* 3, `groupby` sobre columnas categóricas usa `observed=True` por defecto: solo aparecen las combinaciones que existen en los datos.

* La siguiente celda calculará varias estadísticas de la masa por especie con **agregaciones con nombre**.

In [ ]:
pinguinos.groupby("species")["body_mass_g"].agg(
    n="count",
    media="mean",
    mediana="median",
    desv_est="std",
).round(1)

* La siguiente celda usará `pivot_table` para la aleta media por especie (filas) y sexo (columnas), con totales.

In [ ]:
pinguinos.pivot_table(index="species", columns="sex", values="flipper_length_mm",
                      aggfunc="mean", margins=True, margins_name="Total").round(1)

* La siguiente celda usará `crosstab` para contar pingüinos por isla y especie.

In [ ]:
pd.crosstab(pinguinos["island"], pinguinos["species"], margins=True, margins_name="Total")

* La siguiente celda pasará las cuatro medidas a formato largo con `melt`.

In [ ]:
largo = pinguinos.melt(id_vars=["species", "sex"], value_vars=medidas,
                       var_name="medida", value_name="valor")
largo.head(), largo.shape

### 🟢 Ejercicio 1 · Masa por especie y sexo.

Con agregaciones con nombre, calcula para `body_mass_g` por **especie y sexo**: `n`, `media`, `desv_est` y `mediana`. ¿En qué especie es mayor la diferencia entre machos y hembras? (pista: `unstack("sex")` y una resta).

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 2 · Proporciones con `crosstab`.

1. Calcula la proporción de cada especie **dentro de cada isla** (`normalize="index"`) y la proporción de cada isla **dentro de cada especie** (`normalize="columns"`). ¿Qué pregunta responde cada tabla?
2. ¿Por qué no se puede comparar "la masa media por isla" sin pensar en la especie?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 3 · Formato largo y facetas.

Con `largo`, dibuja con `sns.catplot` un diagrama de caja de `valor` por especie, con una faceta por `medida` (`col="medida"`, `sharey=False`, `col_wrap=2`). ¿Qué medida separa mejor a los Gentoo de las otras dos especies?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 1 · Supervivencia en el Titanic por clase y sexo.

Con `titanic.csv`, construye con `pivot_table` la tasa de supervivencia por `class` (filas, en orden First, Second, Third) y `sex` (columnas), con márgenes. Después obtén la misma tabla con `crosstab(..., values=..., aggfunc="mean")`. ¿Qué variable parece pesar más en la supervivencia?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## 2. Relaciones entre variables (25 min).

| Medida | Mide | Rango | Sensible a |
|---|---|---|---|
| **Pearson** (*r*) | Asociación **lineal** | −1 a 1 | Atípicos; relaciones no lineales |
| **Spearman** (ρ) | Asociación **monótona** (Pearson sobre los rangos) | −1 a 1 | Menos sensible a atípicos |

Ambas se calculan con `df.corr(method="pearson")` o `method="spearman"`, o con `stats.pearsonr` y `stats.spearmanr`. Para dos variables **categóricas** se usa una tabla de contingencia (y en la sesión 6, la prueba χ²).

* La siguiente celda calculará Pearson y Spearman entre la cuenta y la propina.

In [ ]:
{"pearson": round(float(propinas["total_bill"].corr(propinas["tip"])), 3),
 "spearman": round(float(propinas["total_bill"].corr(propinas["tip"], method="spearman")), 3)}

* La siguiente celda dibujará la relación con su recta de regresión.

In [ ]:
ax = sns.regplot(propinas, x="total_bill", y="tip", scatter_kws={"alpha": 0.5})
ax.set(xlabel="cuenta (USD)", ylabel="propina (USD)", title="A mayor cuenta, mayor propina (r = 0.68)")
plt.show()

**Ejemplo:** Pearson y Spearman difieren cuando la relación es monótona pero no lineal, o cuando hay atípicos. Se simularán ambos casos.

* La siguiente celda comparará ambas correlaciones en una relación exponencial y en datos sin relación con un solo atípico.

In [ ]:
x = np.arange(1, 21, dtype=float)
exponencial = pd.DataFrame({"x": x, "y": np.exp(x / 3)})
sin_relacion = pd.DataFrame({"x": rng.normal(size=30), "y": rng.normal(size=30)})
sin_relacion.loc[30] = [8.0, 8.0]
pd.DataFrame({
    "exponencial": [exponencial.corr().iloc[0, 1], exponencial.corr("spearman").iloc[0, 1]],
    "ruido + 1 atípico": [sin_relacion.corr().iloc[0, 1], sin_relacion.corr("spearman").iloc[0, 1]],
}, index=["pearson", "spearman"]).round(2)

En la relación exponencial, Spearman vale 1 (la relación es perfectamente monótona) y Pearson solo 0.8 (no es lineal). En los datos sin relación, **un solo punto** lleva a Pearson a 0.72, mientras que Spearman sigue cerca de 0. Conviene calcular ambas y, sobre todo, **graficar**.

### 2.1 Correlación no es causalidad.

Una correlación puede deberse a:
* **Causalidad directa** (X causa Y).
* **Causalidad inversa** (Y causa X).
* Un **confusor**: una tercera variable Z causa ambas.
* **Selección** de la muestra o **azar**.

**Ejemplo:** se simulará un año de datos en el que la **temperatura** aumenta tanto la venta de helados como los ahogamientos. Helados y ahogamientos no se causan entre sí.

* La siguiente celda simulará 365 días de temperatura, venta de helados y ahogamientos.

In [ ]:
temperatura = rng.normal(22, 6, size=365)
dias = pd.DataFrame({
    "temperatura": temperatura,
    "helados": 50 + 8 * temperatura + rng.normal(0, 20, size=365),
    "ahogamientos": rng.poisson(np.clip(0.3 * temperatura - 3, 0.1, None)),
})
dias.corr().round(2)

* La siguiente celda quitará a helados y ahogamientos la parte que explica la temperatura (residuos de una recta ajustada con `np.polyfit`) y correlacionará lo que queda.

In [ ]:
def residuos(y, x):
    pendiente, ordenada = np.polyfit(x, y, deg=1)
    return y - (ordenada + pendiente * x)

round(float(np.corrcoef(residuos(dias["helados"], dias["temperatura"]),
                        residuos(dias["ahogamientos"], dias["temperatura"]))[0, 1]), 2)

Una vez descontado el efecto de la temperatura, la correlación entre helados y ahogamientos prácticamente desaparece: la temperatura era el **confusor**. Controlar por una variable significa comparar observaciones con el mismo valor de esa variable; los residuos son una forma de hacerlo y la paradoja de Simpson, a continuación, muestra otra (comparar **dentro** de cada grupo).

### 2.2 La paradoja de Simpson.

Una tendencia que aparece en todos los grupos **se invierte** (o desaparece) al juntar los grupos, porque los grupos tienen centros distintos. Es un caso extremo de confusión, y `penguins` tiene un ejemplo clásico que se resolverá en el ejercicio 5.

### 🟢 Ejercicio 4 · Matriz de correlaciones.

Calcula las matrices de Pearson y de Spearman de las cuatro medidas de `pinguinos` y dibuja la de Pearson como mapa de calor (`sns.heatmap(..., annot=True, vmin=-1, vmax=1, cmap="vlag")`). ¿Qué par está más correlacionado? ¿Hay alguna correlación negativa?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 5 · La paradoja de Simpson en los picos.

Calcula la correlación de Pearson entre `bill_length_mm` y `bill_depth_mm`:
1. Con todos los pingüinos juntos.
2. Dentro de cada especie (pista: `groupby("species")` y `apply` con una función que calcule la correlación).

¿Qué le pasa al signo? ¿Cuál de las dos respuestas describe mejor la relación entre largo y profundidad del pico?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 2 · ¿Las cuentas grandes dejan menos propina porcentual?

Calcula Pearson y Spearman entre `total_bill` y `tip_pct`, con y sin la cuenta de propina máxima (71%). Grafica la relación. ¿Qué medida es más estable? ¿Qué podría explicar la relación?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## 3. Visualización efectiva (30 min).

### 3.1 Elegir la gráfica según la pregunta.

| Pregunta | Gráficas recomendadas | Evitar |
|---|---|---|
| ¿Cómo se distribuye una variable? | Histograma, KDE, ECDF | Pastel |
| ¿Cómo se compara una numérica entre grupos? | Diagrama de caja, violín, puntos con intervalo, *strip plot* | Barras de medias sin dispersión |
| ¿Cómo se relacionan dos numéricas? | Dispersión (con color por grupo, recta o suavizado) | Líneas que unan puntos no ordenados |
| ¿Cómo se reparte un total? | Barras (apiladas o al 100%) | Pastel con muchas categorías, 3D |
| ¿Cómo cambia en el tiempo? | Líneas | Barras con muchas fechas |

### 3.2 Reglas prácticas.

* **Escalas:** las barras empiezan en cero; para datos multiplicativos, escala logarítmica.
* **Facetas** (`col=`, `row=`) en lugar de amontonar muchos grupos en un solo panel.
* **Color** con significado: una paleta cualitativa para categorías, secuencial para cantidades y divergente para valores alrededor de un centro. Mantener el mismo color para el mismo grupo en todas las gráficas y preferir paletas aptas para daltonismo.
* **Rótulos:** ejes con unidades y un **título que diga el hallazgo**, no solo las variables.
* **Anotaciones** para señalar lo importante.

### 3.3 *Matplotlib* vs. *Seaborn*.

[*Seaborn*](https://seaborn.pydata.org) produce gráficas estadísticas con poco código a partir de un *dataframe* en formato largo (agrupa, colorea, faceta y calcula intervalos). [*Matplotlib*](https://matplotlib.org) da control total de cada elemento. En la práctica se combinan: las funciones de *Seaborn* devuelven un objeto `Axes` de *Matplotlib* que se puede seguir ajustando.

**Ejemplo:** la misma información (masa media por especie) en una gráfica engañosa y en una gráfica efectiva.

* La siguiente celda dibujará ambas versiones lado a lado.

In [ ]:
medias_masa = pinguinos.groupby("species")["body_mass_g"].mean()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(medias_masa.index.astype(str), medias_masa, color=["red", "yellow", "lime"])
axes[0].set_ylim(3600, 5200)
axes[0].set_title("masa")

sns.stripplot(pinguinos, x="species", y="body_mass_g", hue="species", alpha=0.35, legend=False, ax=axes[1])
sns.pointplot(pinguinos, x="species", y="body_mass_g", color="black", linestyle="none",
              errorbar="sd", markers="D", ax=axes[1])
axes[1].set(xlabel="especie", ylabel="masa corporal (g)",
            title="Los Gentoo pesan unos 1,350 g más que Adelie y Chinstrap")
plt.tight_layout()
plt.show()

La versión de la izquierda **corta el eje** (las barras parecen indicar que Gentoo pesa varias veces más), usa colores sin significado y no tiene unidades ni título informativo. La de la derecha muestra **todos los datos**, la media con ± una desviación estándar, unidades y un título con el hallazgo.

* La siguiente celda mostrará facetas: la relación aleta–masa por isla, coloreada por especie.

In [ ]:
grafica = sns.relplot(pinguinos, x="flipper_length_mm", y="body_mass_g", hue="species", col="island",
                      height=3.2, aspect=1, alpha=0.7)
grafica.set_axis_labels("aleta (mm)", "masa (g)")
grafica.set_titles("Isla {col_name}")
plt.show()

* La siguiente celda agregará una anotación con *Matplotlib* sobre una gráfica de *Seaborn*.

In [ ]:
ax = sns.scatterplot(pinguinos, x="flipper_length_mm", y="body_mass_g", hue="species", alpha=0.6)
mas_pesado = pinguinos.loc[pinguinos["body_mass_g"].idxmax()]
ax.annotate(f"El más pesado: {mas_pesado['body_mass_g']:,} g",
            xy=(mas_pesado["flipper_length_mm"], mas_pesado["body_mass_g"]),
            xytext=(185, 6000), arrowprops={"arrowstyle": "->"})
ax.set(xlabel="aleta (mm)", ylabel="masa (g)", title="Aleta y masa crecen juntas en las tres especies")
plt.show()

### 🔴 Reto 3 · Rescata una gráfica.

La siguiente celda dibuja una gráfica de pastel de la cantidad de pingüinos por isla y especie. Identifica al menos tres problemas y rehaz la gráfica para responder con claridad "¿qué especies hay en cada isla y cuántas?".

* La siguiente celda dibujará la gráfica de pastel original.

In [ ]:
conteos = pinguinos.groupby(["island", "species"]).size()
conteos.plot.pie(figsize=(5, 5), autopct="%.0f%%", ylabel="")
plt.show()

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## 4. Ejercicio de la sesión (20 min).

### 🟢 Ejercicio 6 · La gráfica final de la paradoja de Simpson.

Produce **una sola gráfica** que muestre la paradoja del ejercicio 5: dispersión de `bill_length_mm` vs. `bill_depth_mm` coloreada por especie, una recta por especie, la recta global (por ejemplo, negra y punteada), ejes con unidades, la correlación de cada especie y la global en la leyenda o como anotación, y un título que diga el hallazgo.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

### 🟢 Ejercicio 7 · Propina porcentual por día y horario.

1. Construye una tabla de `tip_pct` por `day` (filas) y `time` (columnas) con la **media**, la **mediana** y el **número de cuentas** (pista: `pivot_table(..., aggfunc=["mean", "median", "count"])`).
2. Haz una gráfica que compare `tip_pct` por día y horario mostrando la dispersión, no solo la media.
3. ¿Qué celdas de la tabla están vacías o casi vacías? ¿Qué implica eso para comparar "comida vs. cena"?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 8 · Correlación y causalidad.

Para cada afirmación, indica si es correcta y, si no, cómo la reformularías:
1. "La correlación entre aleta y masa es 0.87, así que las aletas largas hacen que los pingüinos pesen más."
2. "Las mesas que piden en sábado dejan menos propina porcentual porque el sábado la gente es menos generosa."
3. "En el Titanic, viajar en primera clase se **asocia** con una mayor supervivencia; parte de esa asociación podría explicarse por la ubicación de los camarotes y la proporción de mujeres en cada clase."

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 4 · ¿Qué par de medidas separa mejor a las especies?

Dibuja `sns.pairplot` de las cuatro medidas coloreado por especie (con KDE en la diagonal). ¿Con qué par de variables se separarían las tres especies casi sin traslape? Justifica con la gráfica.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## 5. Evaluación semanal 1 (10 min).

**Publicación:** hoy, jueves 22 de octubre, 8:30 pm · **Cierre:** domingo 25 de octubre, 11:59 pm (CDMX). Entregas tardías: hasta 48 h con 10% de penalización.

**Composición (100 puntos):**
* **Cuestionario (40 puntos):** Google Forms en modo examen, 10 preguntas de 4 puntos, un intento. Cubre los Módulos I y II.
* ***Notebook* (60 puntos):** tarea de Classroom con `eval1_eda_descriptivo.ipynb`. Se entrega el `.ipynb` **ejecutado** y el enlace al repositorio o *Codespace*.

**Consigna del *notebook*:** con `tips`, (1) auditar calidad y tipos de variables; (2) crear la variable `tip_pct`; (3) describir su distribución y sus atípicos; (4) comparar `tip_pct` por `day`, `time` y `smoker` con tablas agregadas; (5) producir 2 gráficas efectivas; (6) escribir 3 conclusiones descriptivas sin afirmaciones causales.

**Rúbrica del *notebook* (60 puntos):**

| Criterio | Puntos | Excelente |
|---|---|---|
| Calidad y tipos de variables | 10 | Auditoría completa (duplicados, faltantes, `dtypes`) y justificada |
| Descripción y atípicos | 12 | Medidas adecuadas a la forma de la distribución; atípicos identificados y discutidos |
| Agregación y comparación | 12 | Tablas correctas y bien elegidas para las 3 variables |
| Visualizaciones | 12 | Tipo adecuado, ejes y unidades rotulados, mensaje claro |
| Conclusiones | 8 | Respaldadas por los datos y sin lenguaje causal |
| Reproducibilidad | 6 | El *notebook* corre de principio a fin y está ordenado |

**Nota:** los *notebooks* 01 a 04 contienen todas las herramientas necesarias. Antes de entregar, conviene reiniciar el kernel y ejecutar todo (**Restart & Run All**) para comprobar que el *notebook* corre de principio a fin.

---
## ✅ Cierre.

**Hoy vimos:**
* Agregación con `groupby`/`agg` (agregaciones con nombre), `pivot_table`, `crosstab` y `melt`.
* Correlaciones de Pearson y Spearman, y cuándo difieren.
* Correlación no es causalidad: confusores y la **paradoja de Simpson** en los picos de los pingüinos.
* Visualización efectiva: elegir la gráfica, escalas honestas, facetas, color, rótulos y anotaciones, combinando *Matplotlib* y *Seaborn*.

**Práctica en Classroom:** la **Evaluación semanal 1** cierra el domingo 25 de octubre a las 11:59 pm (CDMX).

**Próxima sesión (lun 26 oct):** incertidumbre: muestreo, error estándar, intervalos de confianza y *bootstrap* (`05_incertidumbre_bootstrap.ipynb`). Se publica la consigna del Proyecto integrador.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>